# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thar-26/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
!git clone https://github.com/thar-26/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 124, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 124 (delta 39), reused 85 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (124/124), 1.84 MiB | 8.00 MiB/s, done.
Resolving deltas: 100% (39/39), done.
/content/flyrank-ml-internship/flyrank-ml-internship


## 1. My lane as an ML task (type)

My lane is Refresh / Content Opportunity Scoring, and I frame it primarily as a ranking task. The goal is not simply to predict whether a page is declining, but to rank pages by priority so that reviewers can decide which pages to inspect first. Each page receives a priority score, and pages with higher scores appear earlier in the review queue. This matches the real decision because reviewer time is limited and only a small number of pages can be inspected first.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

For the starter dataset, I will use is_declining_label as a proxy target for whether a page should receive attention. This label is derived from trend_direction and trend_pct, so it is a defined rule rather than an independently observed future outcome. Because of this, trend_direction and trend_pct must not be used as input features.

For the later capstone work, I want to improve the target by using time-separated warehouse data, where earlier page signals are used to rank pages based on a later observed decline or recovery outcome. This would better match the real decision-support problem and reduce the risk of simply learning the rule that created the starter label.

In [12]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Starter proxy label: a page is declining when trend_direction is "down"
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(df["is_declining_label"].value_counts())
print("\nTarget rate:")
print(f"{df['is_declining_label'].mean():.3f}")


is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target rate:
0.542


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My primary success metric is Precision@50. The output of this project is a ranked review queue, so I care about how many of the first 50 recommended pages are relevant according to the proxy target. A higher Precision@50 means that limited reviewer time is focused on more useful candidates.

For this starter dataset, I will compare learned ranking methods with the hand-written baseline instead of choosing an arbitrary performance threshold. A useful result would be a measured improvement over the baseline on held-out clients. From the starter pipeline, the hand-written rule achieved Precision@50 of 0.240, while the random forest achieved 0.740 under client-holdout validation.

In [13]:
baseline_p50 = 0.240
model_p50 = 0.740

improvement = model_p50 / baseline_p50

print(f"Hand-written baseline Precision@50: {baseline_p50:.3f}")
print(f"Random forest Precision@50: {model_p50:.3f}")
print(f"Relative improvement: {improvement:.1f}x")


Hand-written baseline Precision@50: 0.240
Random forest Precision@50: 0.740
Relative improvement: 3.1x


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is one content page. Each row represents one pseudonymized content item and contains content, search performance, engagement, and age-related signals that may help prioritize the page for review. The identifiers are used only for grouping and splitting, not as model features.

The dataframe below shows the page-level unit of analysis and a small slice of candidate features together with the proxy target.

In [14]:
page_slice = df[
    [
        "content_id",
        "impressions_90d",
        "days_since_last_update",
        "content_age_days",
        "avg_position",
        "ctr",
        "engagement_rate",
        "is_declining_label",
    ]
].copy()

print(f"Rows: {page_slice.shape[0]:,}")
print(f"Columns shown: {page_slice.shape[1]}")
print("One row = one content page")

page_slice.head()

Rows: 30,000
Columns shown: 8
One row = one content page


,content_id,impressions_90d,days_since_last_update,content_age_days,avg_position,ctr,engagement_rate,is_declining_label
0,content_304f48230142,3803,20,187,10.6,0.76,5.88,1
1,content_a1fb4e703a9e,15320,25,445,20.3,0.05,0.00,1
2,content_9aa793d4d895,12581,20,141,36.5,0.09,0.00,1
3,content_331d6c4de07b,11751,22,463,6.2,0.49,1.28,0
4,content_d99b7a2d90ca,19140,14,263,44.0,0.13,0.00,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule can prioritize pages using one or two thresholds, but the review decision may depend on several interacting signals. For example, the importance of content age may differ depending on impressions, average position, CTR, engagement, and other page characteristics. These relationships may be difficult to represent with a single if-statement.

ML can learn combinations of signals and produce a ranking score that can be compared with a transparent rule baseline. However, ML only earns its place if it improves the ranking on held-out clients and produces useful decision support. The final output should remain a ranked queue for human review rather than an automatic decision to refresh a page.

In [15]:
candidate_features = [
    "impressions_90d",
    "days_since_last_update",
    "content_age_days",
    "avg_position",
    "ctr",
    "engagement_rate",
]

print(f"Number of candidate signals shown: {len(candidate_features)}")
print("\nCandidate signals:")
for feature in candidate_features:
    print("-", feature)

print("\nThese signals can interact, so the learned model should be compared")
print("against a simple fixed-rule baseline on held-out clients.")


Number of candidate signals shown: 6

Candidate signals:
- impressions_90d
- days_since_last_update
- content_age_days
- avg_position
- ctr
- engagement_rate

These signals can interact, so the learned model should be compared
against a simple fixed-rule baseline on held-out clients.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.